# Regresión logística en Python
## Caso: abandono de producto financiero

**Curso:** Machine Learning en Economía y Finanzas · FCEA · Clase 9 · 2026-2

En este taller el desenlace es **binario**: el cliente `permanece` (0) o `abandona` (1) la relación con el banco. La estimación se realiza mediante un modelo de regresión logística con `statsmodels` (`logit` / `glm` binomial).


Cuaderno para **Colab** o local. El CSV es `abandono_producto_financiero.csv` (ubicado junto al cuaderno en la raíz del repositorio.).

Cada fila es un cliente observado con desenlace observado: `abandono` = 1 (el cliente cerró la relación) o `abandono` = 0 (el cliente permaneció). El modelo estima $P(y=1 \mid x)$ con $y=1$ si `Abandono` = `1`.

**Respuestas — Punto 1**

**1.** La base cruda tiene **14 variables** (10000 clientes). Tras eliminar `numero_fila`, `id_cliente` y `apellido` quedan **11**: 10 variables explicativas y la respuesta `abandono`.

**2.** Hay **6 cuantitativas**: `puntaje_crediticio`, `edad`, `antiguedad`, `saldo`, `numero_productos` y `salario_estimado`. `numero_productos` es un conteo entero, pero el enunciado lo trata como continua en el logit.

**3.** Hay **5 categóricas**: `pais` y `sexo` (categóricas), `tiene_tarjeta` y `miembro_activo` (binarias 0/1) y la respuesta `abandono`.

**4.** En la fórmula, `C(pais)` y `C(sexo)` hacen que `statsmodels` cree las dummies automáticamente y deje el primer nivel como referencia. Por eso se fija el orden con `pd.Categorical`: así los coeficientes se leen contra Francia y Mujer. `pais` genera 2 dummies y `sexo` genera 1. Las binarias `tiene_tarjeta` y `miembro_activo` ya son 0/1 y entran directamente, sin `C()`. Habría que omitir un nivel por factor para evitar colinealidad perfecta, con más código y más riesgo de error.

In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from IPython.display import display

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = False

CSV = "abandono_producto_financiero.csv"
if not Path(CSV).exists():
    raise FileNotFoundError(
        f"No está {CSV}. Déjelo en la misma carpeta que este cuaderno (raíz del repositorio)."
    )

In [7]:
raw = pd.read_csv(CSV, encoding="utf-8-sig")
raw.head()

,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
0,1,15634602,Hargrave,619,Francia,Mujer,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,España,Mujer,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,Francia,Mujer,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,Francia,Mujer,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,España,Mujer,43,2,125510.82,1,1,1,79084.10,0


In [8]:
raw.tail()

,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
9995,9996,15606229,Obijiaku,771,Francia,Hombre,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,Francia,Hombre,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,Francia,Mujer,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Alemania,Hombre,42,3,75075.31,2,1,0,92888.52,1
9999,10000,15628319,Walker,792,Francia,Mujer,28,4,130142.79,1,1,0,38190.78,0


In [9]:
print(raw.columns.tolist())
print("shape:", raw.shape)

['numero_fila', 'id_cliente', 'apellido', 'puntaje_crediticio', 'pais', 'sexo', 'edad', 'antiguedad', 'saldo', 'numero_productos', 'tiene_tarjeta', 'miembro_activo', 'salario_estimado', 'abandono']
shape: (10000, 14)


In [10]:
raw.describe(include="all")

,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
count,10000.00000,1.000000e+04,10000,10000.000000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
unique,NaN,NaN,2932,NaN,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,Smith,NaN,Francia,Hombre,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,32,NaN,5014,5457,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,5000.50000,1.569094e+07,NaN,650.528800,NaN,NaN,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,NaN,96.653299,NaN,NaN,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,NaN,350.000000,NaN,NaN,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,NaN,584.000000,NaN,NaN,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,NaN,652.000000,NaN,NaN,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,NaN,718.000000,NaN,NaN,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000


Se eliminan los tres identificadores. `pais` y `sexo` pasan a `category` con los niveles del diccionario **en ese orden**, para que `C(pais)` y `C(sexo)` usen como referencia Francia y Mujer. `abandono` queda como entero 0/1.

In [11]:
df = raw.drop(columns=["numero_fila", "id_cliente", "apellido"]).copy()

df["pais"] = pd.Categorical(df["pais"], categories=["Francia", "Alemania", "España"])
df["sexo"] = pd.Categorical(df["sexo"], categories=["Mujer", "Hombre"])
df["abandono"] = df["abandono"].astype(int)

print("Referencia de pais:", df["pais"].cat.categories[0])
print("Referencia de sexo:", df["sexo"].cat.categories[0])
print("Valores faltantes en toda la base:", int(df.isna().sum().sum()))
df.dtypes

Referencia de pais: Francia
Referencia de sexo: Mujer
Valores faltantes en toda la base: 0


puntaje_crediticio       int64
pais                  category
sexo                  category
edad                     int64
antiguedad               int64
saldo                  float64
numero_productos         int64
tiene_tarjeta            int64
miembro_activo           int64
salario_estimado       float64
abandono                 int64
dtype: object

In [12]:
cuantitativas = ["puntaje_crediticio", "edad", "antiguedad", "saldo", "numero_productos", "salario_estimado"]
categoricas = ["pais", "sexo", "tiene_tarjeta", "miembro_activo", "abandono"]

print("Variables en la base cruda:      ", raw.shape[1])
print("Variables tras eliminar los IDs: ", df.shape[1], "(10 explicativas + la respuesta)")
print("Cuantitativas:                   ", len(cuantitativas), cuantitativas)
print("Categóricas o binarias:          ", len(categoricas), categoricas)

Variables en la base cruda:       14
Variables tras eliminar los IDs:  11 (10 explicativas + la respuesta)
Cuantitativas:                    6 ['puntaje_crediticio', 'edad', 'antiguedad', 'saldo', 'numero_productos', 'salario_estimado']
Categóricas o binarias:           5 ['pais', 'sexo', 'tiene_tarjeta', 'miembro_activo', 'abandono']


**Respuestas — Punto 1**

**1.** La base cruda tiene **14 variables** (10000 clientes). Tras eliminar `numero_fila`, `id_cliente` y `apellido` quedan **11**: 10 variables explicativas y la respuesta `abandono`.

**2.** Hay **6 cuantitativas**: `puntaje_crediticio`, `edad`, `antiguedad`, `saldo`, `numero_productos` y `salario_estimado`. `numero_productos` es un conteo entero, pero el enunciado lo trata como continua en el logit.

**3.** Hay **5 categóricas**: `pais` y `sexo` (categóricas), `tiene_tarjeta` y `miembro_activo` (binarias 0/1) y la respuesta `abandono`.

**4.** En la fórmula, `C(pais)` y `C(sexo)` hacen que `statsmodels` cree las dummies automáticamente y deje el primer nivel como referencia. Por eso se fija el orden con `pd.Categorical`: así los coeficientes se leen contra Francia y Mujer. `pais` genera 2 dummies y `sexo` genera 1. Las binarias `tiene_tarjeta` y `miembro_activo` ya son 0/1 y entran directamente, sin `C()`. Habría que omitir un nivel por factor para evitar colinealidad perfecta, con más código y más riesgo de error.